# Load data

In [1]:
import pandas as pd
df = df = pd.read_csv("records1.csv")
print(df.head())

                       narration  mode    type    category  subcategory
0           FD interest credited  NEFT  Credit  investment  fd_interest
1                      FD INT CR  NEFT  Credit  investment  fd_interest
2  Fixed deposit interest payout  NEFT  Credit  investment  fd_interest
3                 Interest on FD  NEFT  Credit  investment  fd_interest
4                FD ROI credited  NEFT  Credit  investment  fd_interest


# Create input_text

In [2]:
df["input_text"] = (
    "narration: " + df["narration"].astype(str) +
    " | mode: " + df["mode"].astype(str) +
    " | type: " + df["type"].astype(str)
)

## Lowercase

In [3]:
df["input_text"] = df["input_text"].str.lower()

# Create label (Ground Truth)

In [4]:
df["label"] = df["category"] + "_" + df["subcategory"]

In [5]:
print(df[["input_text", "label"]].head())

                                          input_text                   label
0  narration: fd interest credited | mode: neft |...  investment_fd_interest
1   narration: fd int cr | mode: neft | type: credit  investment_fd_interest
2  narration: fixed deposit interest payout | mod...  investment_fd_interest
3  narration: interest on fd | mode: neft | type:...  investment_fd_interest
4  narration: fd roi credited | mode: neft | type...  investment_fd_interest


In [6]:
from datasets import Dataset

df["input_text"] = (
    df["narration"] + " " + df["mode"] + " " + df["type"]
)

df["input_text"] = df["input_text"].str.lower()

dataset = Dataset.from_pandas(df[["input_text", "label"]])

In [7]:
labels = list(df["label"].unique())
label2id = {l: i for i, l in enumerate(labels)}
id2label = {i: l for l, i in label2id.items()}

def encode(example):
    example["label"] = label2id[example["label"]]
    return example

dataset = dataset.map(encode)

Map:   0%|          | 0/104 [00:00<?, ? examples/s]

In [8]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize(example):
    return tokenizer(example["input_text"], truncation=True, padding="max_length")

dataset = dataset.map(tokenize)

Map:   0%|          | 0/104 [00:00<?, ? examples/s]

In [9]:
dataset = dataset.train_test_split(test_size=0.2)

train_dataset = dataset["train"]
test_dataset = dataset["test"]

In [10]:
import sys
!{sys.executable} -m pip install --upgrade accelerate transformers torch

     ---------------------------------------- 10.2/10.2 MB 1.8 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.5.1
    Uninstalling transformers-5.5.1:
      Successfully uninstalled transformers-5.5.1


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gliner 0.2.26 requires transformers<5.2.0,>=4.51.3, but you have transformers 5.5.3 which is incompatible.

[notice] A new release of pip available: 22.2.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id
)

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=4,
    num_train_epochs=6,
    logging_steps=10,
    save_strategy="no",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

trainer.train()

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
c:\Users\ACER\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  supe

Step,Training Loss
10,2.701654
20,2.586775
30,2.349247
40,2.127423
50,1.761073
60,1.703879
70,1.344613
80,1.276799
90,1.272411
100,0.902191


TrainOutput(global_step=126, training_loss=1.6440149072616819, metrics={'train_runtime': 829.7058, 'train_samples_per_second': 0.6, 'train_steps_per_second': 0.152, 'total_flos': 65984058501120.0, 'train_loss': 1.6440149072616819, 'epoch': 6.0})

In [12]:
import torch
import torch.nn.functional as F

def predict(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True)
    outputs = model(**inputs)

    probs = F.softmax(outputs.logits, dim=1)

    confidence = torch.max(probs).item()
    pred_id = torch.argmax(probs).item()

    label = id2label[pred_id]
    category, subcategory = label.split("_", 1)

    return category, subcategory, round(confidence, 2)

def confidence_level(conf):
    if conf >= 0.65:
        return "High"
    elif conf >= 0.4:
        return "Medium"
    else:
        return "Low"

In [13]:
results = []

for i in range(len(df)):
    narration = df["narration"].iloc[i]
    mode = df["mode"].iloc[i]
    txn_type = df["type"].iloc[i]

    # same input used for model
    text = df["input_text"].iloc[i]

    pred_category, pred_subcategory, confidence = predict(text)

    results.append({
        "narration": narration,
        "mode": mode,
        "type": txn_type,
        "category": df["category"].iloc[i],
        "subcategory": df["subcategory"].iloc[i],
        "predicted_category": pred_category,
        "predicted_subcategory": pred_subcategory,
        "confidence": confidence,
        "confidence_percent": f"{round(confidence*100,1)}%",
        "confidence_level": confidence_level(confidence),
        "correct": (df["category"].iloc[i] == pred_category) and 
           (df["subcategory"].iloc[i] == pred_subcategory)
    })

result_df = pd.DataFrame(results)

print(result_df.to_string())

                         narration         mode    type    category  subcategory predicted_category predicted_subcategory  confidence confidence_percent confidence_level  correct
0             FD interest credited         NEFT  Credit  investment  fd_interest         investment           fd_interest        0.70              70.0%             High     True
1                        FD INT CR         NEFT  Credit  investment  fd_interest         investment           fd_interest        0.65              65.0%             High     True
2    Fixed deposit interest payout         NEFT  Credit  investment  fd_interest         investment           fd_interest        0.66              66.0%             High     True
3                   Interest on FD         NEFT  Credit  investment  fd_interest         investment           fd_interest        0.70              70.0%             High     True
4                  FD ROI credited         NEFT  Credit  investment  fd_interest         investment      

## | Metric | Meaning      | Use here               |
### | ------ | ------------ | ---------------------- |
### | F1     | balance      | good baseline          |
### | F2     | recall heavy | better for your system |


In [14]:
y_true = result_df["category"] + "_" + result_df["subcategory"]
y_pred = result_df["predicted_category"] + "_" + result_df["predicted_subcategory"]

In [15]:
from sklearn.metrics import f1_score, fbeta_score

f1 = f1_score(y_true, y_pred, average='weighted')
f2 = fbeta_score(y_true, y_pred, beta=2, average='weighted')

print("Overall Metrics:")
print(f"F1 Score: {round(f1,3)}")
print(f"F2 Score: {round(f2,3)}")

Overall Metrics:
F1 Score: 0.786
F2 Score: 0.812


In [16]:
from sklearn.metrics import classification_report
import pandas as pd

report = classification_report(y_true, y_pred, output_dict=True)

report_df = pd.DataFrame(report).transpose()

print(report_df)

                        precision    recall  f1-score     support
expense_bills            1.000000  0.900000  0.947368   10.000000
expense_food             0.666667  1.000000  0.800000   12.000000
expense_health           1.000000  0.500000  0.666667    4.000000
expense_insurance        1.000000  1.000000  1.000000    3.000000
expense_loan             1.000000  1.000000  1.000000    4.000000
expense_shopping         0.846154  1.000000  0.916667   11.000000
expense_transport        1.000000  0.250000  0.400000    4.000000
income_bonus             0.000000  0.000000  0.000000    5.000000
income_cashback          1.000000  1.000000  1.000000    2.000000
income_refund            0.000000  0.000000  0.000000    3.000000
income_salary            0.631579  1.000000  0.774194   12.000000
investment_fd_booking    0.900000  1.000000  0.947368    9.000000
investment_fd_interest   0.928571  1.000000  0.962963   13.000000
investment_mutual_fund   1.000000  1.000000  1.000000    9.000000
investment

c:\Users\ACER\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\ACER\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\ACER\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(averag

In [17]:
from sklearn.metrics import fbeta_score

labels = list(set(y_true))

f2_scores = {}

for label in labels:
    y_true_bin = [1 if y == label else 0 for y in y_true]
    y_pred_bin = [1 if y == label else 0 for y in y_pred]

    f2_scores[label] = fbeta_score(y_true_bin, y_pred_bin, beta=2)

report_df["f2_score"] = report_df.index.map(f2_scores)

In [18]:
report_df = report_df.round(3)
print(report_df)

                        precision  recall  f1-score  support  f2_score
expense_bills               1.000   0.900     0.947   10.000     0.918
expense_food                0.667   1.000     0.800   12.000     0.909
expense_health              1.000   0.500     0.667    4.000     0.556
expense_insurance           1.000   1.000     1.000    3.000     1.000
expense_loan                1.000   1.000     1.000    4.000     1.000
expense_shopping            0.846   1.000     0.917   11.000     0.965
expense_transport           1.000   0.250     0.400    4.000     0.294
income_bonus                0.000   0.000     0.000    5.000     0.000
income_cashback             1.000   1.000     1.000    2.000     1.000
income_refund               0.000   0.000     0.000    3.000     0.000
income_salary               0.632   1.000     0.774   12.000     0.896
investment_fd_booking       0.900   1.000     0.947    9.000     0.978
investment_fd_interest      0.929   1.000     0.963   13.000     0.985
invest